# EA2 — Despliegue y gobierno de una infraestructura de datos en la nube

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *56* |
| **Integrantes** | *Paula Andrea Celis Cano* |
| **Caso de estudio** | *(Wanderbricks)* |
| **Fecha de entrega** | domingo 20 de septiembre |
| **🎥 Enlace al video** | *(pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

Wanderbricks es una empresa que maneja información relacionada con reservas de alojamientos. Estos datos contienen información sobre las reservas, usuarios, propiedades, estados de las reservas y valores asociados.

El principal problema es que, al manejar una cantidad importante de información, se necesita organizar y transformar los datos para que puedan ser utilizados de manera sencilla en procesos de análisis y toma de decisiones.

Para solucionar este problema se propone utilizar una arquitectura de datos por capas, utilizando Databricks. Los datos se organizan en tres niveles:

* **Bronze:** almacena los datos provenientes de la fuente con transformaciones mínimas.
* **Plata:** contiene los datos limpiados y transformados.
* **Oro:** contiene información resumida y preparada para el análisis.

Esta organización permite mejorar el manejo de los datos, facilitar su procesamiento y obtener información útil a partir de las reservas de Wanderbricks.

El objetivo de la solución es construir un flujo de datos que permita pasar desde la fuente original hasta información consolidada para análisis, aplicando procesos de ingesta, limpieza, transformación y agregación.



---
## 2. Descripción de los datos

Los datos utilizados en el proyecto corresponden a información de reservas de la plataforma Wanderbricks. La fuente utilizada en Databricks es:

`samples.wanderbricks.bookings`

Esta información contiene datos relacionados con las reservas realizadas por los usuarios y las propiedades reservadas.

Entre los principales campos utilizados se encuentran:

* **booking_id:** identifica de manera única cada reserva.
* **user_id:** identifica al usuario que realizó la reserva.
* **property_id:** identifica la propiedad relacionada con la reserva.
* **status:** indica el estado en el que se encuentra la reserva.
* **total_amount:** representa el valor total asociado a la reserva.

Durante el proceso de transformación también se utilizan y generan algunos campos adicionales:

* **_ingesta_ts:** registra la fecha y hora en que los datos fueron ingresados al proceso.
* **_origen:** permite identificar el origen de los datos.
* **dias_estadia:** representa la cantidad de días de la estadía y se obtiene durante la transformación de los datos.

Los datos pasan por diferentes etapas. Primero se almacenan en la capa **Bronze**, donde se conservan con transformaciones mínimas. Después pasan a **Plata**, donde se realizan validaciones y transformaciones, como el cálculo de `dias_estadia`. Finalmente, en **Oro**, se genera un resumen de las reservas ag_


# 3. Decisions / Design

## 3.1 Arquitectura propuesta

La solución propuesta para Wanderbricks utiliza una arquitectura de datos por capas, implementada en Databricks.

**Flujo de datos:**

**Fuentes → Ingesta → Almacenamiento → Procesamiento → Consumo**

* **Fuentes:** datos de reservas de Wanderbricks, representados por la tabla `samples.wanderbricks.bookings`.
* **Ingesta:** lectura de los datos desde la fuente y agregación de información de control como `_ingesta_ts` y `_origen`.
* **Almacenamiento:** organización de los datos mediante las capas Bronze, Silver y Gold.
* **Procesamiento:** limpieza, transformación y cálculo de nuevas variables como `dias_estadia`.
* **Consumo:** tablas Gold con información resumida que puede ser utilizada para análisis y toma de decisiones.

### Gestión de las capas

| Capa    | Responsable principal  | Descripción                                                                   |
| ------- | ---------------------- | ----------------------------------------------------------------------------- |
| Fuentes | Proveedor              | Los datos de origen son proporcionados por la plataforma.                     |
| Ingesta | Proveedor / plataforma | Databricks proporciona los servicios necesarios para trabajar con los datos.  |
| Bronze  | Estudiante             | Se crea y administra la tabla de datos crudos o con transformaciones mínimas. |
| Silver  | Estudiante             | Se realizan las validaciones y transformaciones necesarias.                   |
| Gold    | Estudiante             | Se generan datos agregados preparados para análisis.                          |
| Consumo | Estudiante / usuarios  | Los resultados pueden ser consultados para análisis y reportes.               |

En esta arquitectura, la plataforma administrada reduce la necesidad de instalar y mantener servidores. El estudiante se concentra principalmente en la organización, transformación y análisis de los datos.

---

## 3.2 Organización del entorno

Se creó el catálogo:

`bigdata_grupo56`

Dentro del catálogo se organizaron tres esquemas:

* `bronze`: contiene los datos provenientes de la fuente con transformaciones mínimas.
* `plata`: contiene los datos limpios y transformados.
* `oro`: contiene información agregada y preparada para el consumo analítico.

También se creó un volumen para los datos crudos:

`bigdata_grupo56.bronce.datos_crudos`

Esta organización permite separar las diferentes etapas del procesamiento y facilita la administración, seguridad y trazabilidad de los datos.

---

## 3.3 Matriz de roles y permisos

La siguiente matriz representa cómo se podrían organizar los permisos para tres roles principales: Analista, Data Engineer y Administrador.

| Objeto / Capa  | Analista      | Data Engineer                                    | Administrador                 |
| -------------- | ------------- | ------------------------------------------------ | ----------------------------- |
| Bronze         | Lectura       | Lectura y modificación                           | Control total                 |
| Silver / Plata | Lectura       | Lectura y modificación                           | Control total                 |
| Gold / Oro     | Lectura       | Lectura y modificación                           | Control total                 |
| Catálogo       | Uso           | Uso y administración según necesidad             | Control total                 |
| Permisos       | No administra | Administra los permisos de los objetos asignados | Administra todos los permisos |

### Justificación

El **Analista** necesita principalmente consultar información, por lo que debe tener permisos de lectura sobre las tablas analíticas.

El **Data Engineer** necesita crear, modificar y transformar datos en las diferentes capas, por lo que requiere permisos adicionales de modificación.

El **Administrador** necesita controlar el entorno, administrar permisos y gestionar los diferentes objetos.

Aunque la plataforma utilizada para la actividad tiene limitaciones respecto a la administración de grupos, esta matriz representa el diseño de roles que tendría una implementación completa.

---

## 3.4 Diseño equivalente en IaaS

Si la solución tuviera que implementarse utilizando infraestructura IaaS, sería necesario administrar directamente las máquinas virtuales y los componentes de la plataforma.

### Máquinas virtuales propuestas

| Componente                 | Cantidad | Características aproximadas                  |
| -------------------------- | -------: | -------------------------------------------- |
| Nodo principal             |        1 | 4–8 vCPU, 16–32 GB RAM                       |
| Nodos de procesamiento     |        2 | 4–8 vCPU, 16–32 GB RAM cada uno              |
| Servidor de almacenamiento |        1 | 4 vCPU, 16 GB RAM y almacenamiento ampliable |

### Sistema operativo

Se podría utilizar una distribución Linux, por ejemplo Ubuntu Server, debido a su compatibilidad con herramientas de procesamiento de datos.

### Software

Sería necesario instalar y configurar manualmente:

* Java.
* Apache Spark.
* Python.
* Herramientas de administración de datos.
* Componentes de almacenamiento.
* Herramientas de monitoreo y seguridad.
* Servicios necesarios para ejecutar los procesos ETL.

### Red

Se necesitaría configurar:

* Red virtual.
* Subredes.
* Direcciones IP.
* Reglas de firewall.
* Acceso seguro entre las máquinas.
* Control de acceso para los usuarios.

### Almacenamiento

Se requeriría almacenamiento suficiente para conservar los datos originales, procesados y resultados finales. También sería necesario considerar copias de seguridad y crecimiento futuro de los datos.

### Esfuerzo de operación

El uso de IaaS requiere mayor trabajo administrativo porque el equipo debe encargarse de crear las máquinas, instalar software, actualizar sistemas, configurar seguridad, monitorear recursos y solucionar problemas de infraestructura.

---

## 3.5 Comparación IaaS vs PaaS vs SaaS

| Criterio                          | IaaS                              | PaaS                | SaaS                                       |
| --------------------------------- | --------------------------------- | ------------------- | ------------------------------------------ |
| Control                           | Alto                              | Medio               | Bajo                                       |
| Tiempo para obtener resultados    | Mayor                             | Medio               | Menor                                      |
| Esfuerzo operativo                | Alto                              | Medio               | Bajo                                       |
| Costo inicial                     | Mayor                             | Medio               | Menor                                      |
| Escalabilidad                     | Requiere configuración            | Más sencilla        | Generalmente administrada por el proveedor |
| Gobernanza                        | Mayor responsabilidad del usuario | Compartida          | Principalmente del proveedor               |
| Administración de infraestructura | Usuario                           | Proveedor y usuario | Proveedor                                  |

### Interpretación

**IaaS** proporciona mayor control sobre la infraestructura, pero requiere administrar servidores, redes, almacenamiento y software.

**PaaS** permite concentrarse más en el desarrollo y procesamiento de datos porque el proveedor administra una parte importante de la infraestructura.

**SaaS** ofrece una aplicación lista para utilizar y requiere la menor administración de infraestructura por parte del usuario.

Para el caso de Wanderbricks, una plataforma administrada de datos como la utilizada en esta actividad permite concentrar el trabajo en los datos, sus transformaciones y su análisis, reduciendo la administración directa de servidores.

---

## 3.6 Conclusión de diseño

La arquitectura propuesta organiza los datos de Wanderbricks mediante las capas Bronze, Plata y Oro, permitiendo separar la ingesta, transformación y consumo de la información.

El uso de una plataforma administrada reduce el esfuerzo necesario para instalar y mantener infraestructura, mientras que la organización por capas facilita el control y la trazabilidad de los datos.

El diseño IaaS sería posible, pero requeriría administrar máquinas virtuales, sistemas operativos, redes, almacenamiento, software, seguridad y mantenimiento. Por esta razón, para este caso resulta más importante disponer de una plataforma que permita enfocarse en el procesamiento y análisis de los datos sin tener que administrar toda la infraestructura física o virtual.


---
## 4. Implementación
### 4.1 Organización del entorno

In [0]:
# ==========================================
# EA2 - Organización del entorno Wanderbricks
# ==========================================

CATALOGO = "bigdata_grupo56"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.bronce")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.plata")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.oro")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.bronce.datos_crudos")

print("✅ Organización de Wanderbricks creada")

In [0]:
# Mostrar los esquemas creados
display(
    spark.sql(f"SHOW SCHEMAS IN {CATALOGO}")
)

In [0]:
from pyspark.sql import functions as F

bookings = spark.table("samples.wanderbricks.bookings")

bookings_bronze = (
    bookings
    .withColumn("_ingesta_ts", F.current_timestamp())
    .withColumn("_origen", F.lit("samples.wanderbricks.bookings"))
)

bookings_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bigdata_grupo56.bronce.bookings")

print("✅ Tabla Bronze bookings creada")

In [0]:
display(
    spark.table(f"{CATALOGO}.bronce.bookings")
    .limit(10)
)

In [0]:
cantidad = spark.table(
    f"{CATALOGO}.bronce.bookings"
).count()

print(f"Bronze bookings: {cantidad:,} filas")

In [0]:
from pyspark.sql import functions as F

bookings_bronze = spark.table(
    "bigdata_grupo56.bronce.bookings"
)

bookings_plata = (
    bookings_bronze
    .filter(F.col("booking_id").isNotNull())
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("property_id").isNotNull())
    .withColumn(
        "dias_estadia",
        F.datediff(
            F.col("check_out"),
            F.col("check_in")
        )
    )
)

bookings_plata.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "bigdata_grupo56.plata.bookings"
    )

print("✅ Tabla Plata bookings creada")

In [0]:
bookings_plata = spark.table(
    "bigdata_grupo56.plata.bookings"
)

resumen_oro = (
    bookings_plata
    .groupBy("status")
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(
            F.sum("total_amount"), 2
        ).alias("ingresos_totales")
    )
)

resumen_oro.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "bigdata_grupo56.oro.resumen_reservas"
    )

print("✅ Tabla Oro creada")

In [0]:
bookings_plata = spark.table(
    "bigdata_grupo56.plata.bookings"
)

resumen_oro = (
    bookings_plata
    .groupBy("status")
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(
            F.sum("total_amount"), 2
        ).alias("ingresos_totales")
    )
)

resumen_oro.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "bigdata_grupo56.oro.resumen_reservas"
    )

print("✅ Tabla Oro creada")

### 4.2 Permisos

*Al menos dos sentencias GRANT con niveles distintos sobre objetos distintos.*

In [0]:
CATALOGO = "bigdata_grupo56"

# Permiso para usar el catálogo
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOGO} TO `account users`")

# Permisos sobre ORO
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOGO}.oro TO `account users`")
spark.sql(f"GRANT SELECT ON SCHEMA {CATALOGO}.oro TO `account users`")

# Permisos sobre PLATA
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOGO}.plata TO `account users`")
spark.sql(f"GRANT MODIFY ON SCHEMA {CATALOGO}.plata TO `account users`")

print("✅ Permisos otorgados correctamente")

In [0]:
print("Permisos sobre ORO:")
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOGO}.oro"))

print("Permisos sobre PLATA:")
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOGO}.plata"))

### 4.3 Linaje
### Evidencia del linaje

La siguiente captura muestra el linaje de la tabla `bigdata_grupo56.oro.resumen_reservas` desde el Catalog Explorer. Se observa como origen la tabla `bigdata_grupo56.plata.bookings`.
.
![Captura de pantalla 2026-09-16 210740.png](./Captura de pantalla 2026-09-16 210740.png "Captura de pantalla 2026-09-16 210740.png")

### 4.4 Automatización

*Un Job con al menos dos tareas encadenadas y una programación definida.
Insertar la captura de una ejecución exitosa e indicar el identificador del Job.*

EA2_Tarea_Bronce

In [0]:
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo56"

bookings = spark.table("samples.wanderbricks.bookings")

bookings_bronze = (
    bookings
    .withColumn("_ingesta_ts", F.current_timestamp())
    .withColumn("_origen", F.lit("samples.wanderbricks.bookings"))
)

bookings_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOGO}.bronce.bookings")

print("✅ Tarea Bronze completada")

EA2_Tarea_Plata_Oro

In [0]:
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo56"

# Leer Bronze
bookings_bronze = spark.table(
    f"{CATALOGO}.bronce.bookings"
)

# Crear Plata
bookings_plata = (
    bookings_bronze
    .filter(F.col("booking_id").isNotNull())
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("property_id").isNotNull())
    .withColumn(
        "dias_estadia",
        F.datediff(
            F.col("check_out"),
            F.col("check_in")
        )
    )
)

bookings_plata.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{CATALOGO}.plata.bookings"
    )

# Crear Oro
resumen_oro = (
    bookings_plata
    .groupBy("status")
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(
            F.sum("total_amount"), 2
        ).alias("ingresos_totales")
    )
)

resumen_oro.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"{CATALOGO}.oro.resumen_reservas"
    )

print("✅ Tarea Plata y Oro completada")

Evidencvia del job

![Captura de pantalla 2026-09-18 204851.png](./Captura de pantalla 2026-09-18 204851.png "Captura de pantalla 2026-09-18 204851.png")

---
## 5. Resultados
# 5. Resultados

Durante la implementación se logró construir un flujo de procesamiento de datos utilizando una arquitectura por capas.

### Catálogo y esquemas

Se creó correctamente el catálogo:

`bigdata_grupo56`

con los esquemas:

* `bronze`
* `plata`
* `oro`

También se creó el volumen:

`bigdata_grupo56.bronce.datos_crudos`

### Capa Bronze

Los datos fueron obtenidos desde:

`samples.wanderbricks.bookings`

y almacenados en:

`bigdata_grupo56.bronce.bookings`

Durante la ingesta se agregaron las columnas `_ingesta_ts` y `_origen`, permitiendo identificar información relacionada con el proceso de carga.

### Capa Plata

A partir de Bronze se realizó una transformación para conservar registros con información válida de:

* `booking_id`
* `user_id`
* `property_id`

También se calculó la variable:

`dias_estadia`

Los resultados fueron almacenados en:

`bigdata_grupo56.plata.bookings`

### Capa Oro

Finalmente se creó la tabla:

`bigdata_grupo56.oro.resumen_reservas`

Esta tabla resume las reservas agrupándolas por estado (`status`) y calculando:

* cantidad de reservas;
* suma del valor total de las reservas.

De esta manera, la capa Oro queda preparada para consultas y análisis.

### Permisos

Se ejecutaron instrucciones `GRANT` para asignar permisos diferentes sobre los esquemas Plata y Oro. Posteriormente se utilizó `SHOW GRANTS` para consultar los permisos asignados.

### Automatización

El proceso fue organizado en dos tareas:

1. `EA2_Tarea_Bronce`
2. `EA2_Tarea_Plata_Oro`

La segunda tarea utiliza los datos generados por la primera, formando una dependencia entre ambas.

En conjunto, los resultados muestran un flujo completo desde la fuente de datos hasta una tabla agregada para análisis.


---
## 6. Conclusiones

# 6. Conclusiones

La actividad permitió comprender cómo una plataforma de datos administrada puede utilizarse para construir un flujo completo de procesamiento de información.

La organización mediante las capas Bronze, Plata y Oro permite separar los datos según su nivel de transformación y facilita su posterior análisis.

También se comprobó la importancia de los permisos, ya que mediante `GRANT` es posible controlar qué acciones pueden realizar los usuarios sobre los diferentes objetos de datos.

La implementación de tareas automatizadas permite organizar el procesamiento en diferentes etapas y establecer dependencias entre ellas. Esto facilita la ejecución repetitiva de los procesos sin tener que realizar todos los pasos manualmente.

Finalmente, al comparar IaaS, PaaS y SaaS, se observa que cada modelo proporciona un nivel diferente de control y responsabilidad. Para un proyecto de análisis de datos como Wanderbricks, una solución administrada permite dedicar más esfuerzo al procesamiento y análisis de los datos y menos a la administración de infraestructura.


---
## 7. Reparto del trabajo y uso de IA

## 7.1 Repartición del trabajo

| Integrante              | Actividades realizadas                                                                                                                                                                                                                            |
| ----------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Paula Andrea Celis Cano | Diseño de la arquitectura, creación del catálogo y esquemas, implementación de las capas Bronze, Plata y Oro, configuración de permisos, diseño de la automatización.

## 7.2 Uso de inteligencia artificial

Se utilizó inteligencia artificial como herramienta de apoyo para:

* Comprender los conceptos de arquitectura de datos.
* Revisar la estructura del proyecto.
* Facilitar la comprensión de conceptos relacionados con IaaS, PaaS y SaaS.

La implementación práctica, ejecución de comandos y revisión de los resultados fueron realizadas sobre el entorno de trabajo utilizado para la actividad.


---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Qué parte de esta arquitectura administra el proveedor y cuál administran ustedes?
2. Muestre un GRANT que ejecutó y explique a quién le está dando qué, y por qué.
3. Si tuvieran que montar esto sobre máquinas virtuales, ¿qué sería lo primero que se les complicaría?

---
## ✅ Antes de entregar

- [ ] El diagrama de arquitectura está incluido y descrito
- [ ] Los dos GRANT están ejecutados y el SHOW GRANTS muestra el resultado
- [ ] El Job tiene dos o más tareas, está programado y hay evidencia de ejecución
- [ ] La comparación IaaS/PaaS/SaaS termina en una conclusión, no en una tabla suelta
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todo confirmado en /ea2 y el HTML subido a Canvas